# Transfer learning: adapting TERRA to Visium


Fine-tune a **Xenium-pretrained TERRA model** on **10x Visium** data with
**LoRA**, then embed new Visium sections and identify cell types and states.

Visium differs from TERRA's imaging-based pretraining data (multi-cell spots,
whole-transcriptome, sequencing counts). **Transfer learning** adapts the model
by training only small **LoRA adapters** on the frozen backbone; fast,
data-efficient, and non-destructive.

You will:

1. **Prepare** per-section Visium `AnnData` objects and quality-control them.
2. **Harmonize & tokenize** each section against the pretrained model's vocabulary.
3. **Fine-tune** the pretrained TERRA encoder on Visium with LoRA adapters (I-JEPA objective).
4. **Merge** the LoRA adapters back into the base model to obtain a self-contained embedder.
5. **Embed** Visium sections with the adapted model.
6. **Analyse** the embeddings, mapped back onto the tissue and benchmark against the zero-shot and scGPT-spatial baselines.

:::{warning}
Tokenization, fine-tuning and embedding all require an **NVIDIA GPU**.
:::

## Setup

Everything in this tutorial uses the packaged `terra` API: `terra` for
harmonization, tokenization and embedding, and
`terra.training.finetune_self_supervised` (`finetune_self_supervised` and
`prepare_finetuned_model`) for the LoRA fine-tune and adapter merge.

In [ ]:
# Colab / fresh environment only. Skip if you already installed TERRA per the installation guide.
%pip install terra-st

In [ ]:
import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from terra import download_pretrained, harmonize_adata, tokenize_adata, embed_dataset

sc.settings.set_figure_params(dpi=100, facecolor="white", fontsize=8)

### Configure paths

The pretrained TERRA model is downloaded from the Hugging Face Hub. 

We keep two data streams:

- a **fine-tuning set** (`FT_*`), sections the LoRA adapters are trained on;
- an **inference set** (`ZS_*`), sections we embed with the adapted model and
  analyse (these can be held out from fine-tuning).

In [ ]:
# Pretrained TERRA model (96M) — downloaded from the Hugging Face Hub
PRETRAINED_MODEL_DIR = download_pretrained("lotfollahi-lab/TERRA-96M")

# Working directories — point these at your own data
DATA_DIR = Path("data/visium_embryo")   
RAW_DIR = DATA_DIR / "raw"                # combined / per-section h5ad

# Fine-tuning stream (train the LoRA adapters on these sections)
FT_QC_DIR = DATA_DIR / "finetuning/qc"
FT_HARMONISED_DIR = DATA_DIR / "finetuning/harmonised"
FT_TOKENISED_DIR = DATA_DIR / "finetuning/tokenised"

# Inference / evaluation stream (embed these with the adapted model)
ZS_QC_DIR = DATA_DIR / "inference/qc"
ZS_EMBED_DIR = DATA_DIR / "inference/embeddings"

# Fine-tuning outputs
FINETUNE_CKPT_DIR = DATA_DIR / "finetuned_models"
FINETUNED_MODEL_DIR = FINETUNE_CKPT_DIR / "visium_lora_merged"   

# Scratch cache used during tokenisation
CACHE_DIR = "terra_cache"

for d in [FT_QC_DIR, FT_HARMONISED_DIR, FT_TOKENISED_DIR, ZS_QC_DIR, ZS_EMBED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## Step 1: Prepare and QC the Visium sections

### 1a. Split a combined object into per-section files

TERRA works per **section**. The embryo data ships as one combined object with a
`sample` column identifying each section. We split it, keep the Ensembl IDs,
copy the first two spatial dimensions into `obsm["spatial"]`, and write one
`<sample>_norm.h5ad` per section.

In [ ]:
def process_embryo_sample(adata, sample_name, output_dir):
    # Preserve Ensembl IDs, expose gene symbols on the index if available
    adata.var["ensembl_id"] = adata.var.index
    if "SYMBOL" in adata.var.columns:
        adata.var.index = adata.var["SYMBOL"]
        adata.var.index.name = "gene_name"
    adata.var = adata.var[["ensembl_id"]]

    # TERRA needs 2-D spatial coordinates (microns) in obsm["spatial"]
    adata.obsm["spatial"] = adata.obsm["xyz_um"][:, :2].copy()

    # Keep only this section's uns entries, then write it out
    adata.uns = {k: v for k, v in adata.uns.items() if sample_name in k}
    adata.write_h5ad(os.path.join(output_dir, f"{sample_name}_norm.h5ad"),
                     compression="gzip")


combined = ad.read_h5ad(RAW_DIR / "adata_CS17.h5ad")
for sample_name in combined.obs["sample"].unique():
    section = combined[combined.obs["sample"] == sample_name].copy()
    process_embryo_sample(section, sample_name, str(RAW_DIR))
print(f"Wrote {combined.obs['sample'].nunique()} per-section files to {RAW_DIR}")

:::{note}
TERRA requirements for each section: **raw integer counts** in `X` (or `layers["counts"]`), gene names /
Ensembl IDs in `var`, and 2-D coordinates in `obsm["spatial"]`.
:::

### 1b. Quality-control each section

Standard spot-level QC.

In [ ]:
def qc_section(adata, min_genes=200, max_pct_mt=20):
    # Mitochondrial genes 
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None,
                               log1p=False, inplace=True)

    # Per-section count bounds
    lo = adata.obs["total_counts"].quantile(0.01)
    hi = adata.obs["total_counts"].quantile(0.99)

    n0 = adata.n_obs
    sc.pp.filter_cells(adata, min_counts=lo)
    sc.pp.filter_cells(adata, max_counts=hi)
    sc.pp.filter_cells(adata, min_genes=min_genes)
    adata = adata[adata.obs["pct_counts_mt"] < max_pct_mt].copy()
    print(f"  {n0} -> {adata.n_obs} spots after QC")
    return adata


# QC every prepared section into the fine-tuning QC directory
for h5_file in sorted(RAW_DIR.glob("*_norm.h5ad")):
    if h5_file.name == "adata_CS17.h5ad":
        continue
    sample_name = h5_file.stem.rsplit("_norm", 1)[0]
    print(f"QC {sample_name}")
    adata = qc_section(sc.read_h5ad(h5_file))
    adata.write(FT_QC_DIR / f"{sample_name}_QC_norm.h5ad")
    del adata
    gc.collect()

## Step 2: Harmonize & tokenize

Before the model can read a section it must be **harmonized** (gene names mapped
to the Ensembl IDs used at pretraining, low-quality genes) and
**tokenized** (turned into the cell + neighborhood token sequences the encoder
expects). The harmonization will ensure the Visium data is compatible with the model; it is expected for a significant proportion of genes to be excluded as the model may have never seen them before, due to the smaller panel size of Xenium.

In [ ]:
def harmonize_and_tokenize(h5ad_path, out_dir, model_dir, num_shards=32):
    adata = sc.read_h5ad(h5ad_path)

    # Map gene names -> Ensembl IDs, filter, and expose x/y from obsm['spatial']
    adata = harmonize_adata(adata)

    # Drop spots with missing coordinates
    nan_mask = adata.obs[["x", "y"]].isna().any(axis=1)
    if nan_mask.any():
        adata = adata[~nan_mask].copy()

    # Tokenize against the pretrained model's config + vocabulary (needs a GPU)
    dataset = tokenize_adata(
        adata,
        model_folder_path=model_dir,
        cache_directory_path=CACHE_DIR,
        nproc=4,
        processing_mode="parallel",
    )

    sample_name = Path(h5ad_path).stem
    save_path = Path(out_dir) / f"{sample_name}_tokenised"
    save_path.mkdir(parents=True, exist_ok=True)
    dataset.save_to_disk(str(save_path), num_shards=num_shards)
    print(f"  tokenised -> {save_path}")
    return save_path


for h5_file in sorted(FT_QC_DIR.glob("*.h5ad")):
    print(f"Tokenising {h5_file.stem}")
    harmonize_and_tokenize(h5_file, FT_TOKENISED_DIR, PRETRAINED_MODEL_DIR)

## Step 3: Fine-tune TERRA on Visium with LoRA

Now we adapt the pretrained encoder to Visium. Fine-tuning reuses TERRA's
**I-JEPA** self-supervised objective: a context encoder sees part of each
cell/neighborhood token sequence and a predictor must match the representation
of masked target blocks produced by an EMA **target encoder**. The pretrained
weights are frozen; only the **LoRA adapters** on the attention/MLP projections
are trained, and the target encoder is updated by EMA of the (adapter-merged)
context encoder.

This is set in the `args` dict below:

In [ ]:
# Fine-tuning configuration 
args = {
    "model": {
        "pretrained_checkpoint_path": PRETRAINED_MODEL_DIR,   # Xenium-pretrained TERRA
        "finetune_checkpoint_path": str(FINETUNE_CKPT_DIR),   # where checkpoints are saved
    },
    "data": {
        # every tokenised section from Step 2
        "finetune_dataset": [str(p) for p in sorted(FT_TOKENISED_DIR.glob("*_tokenised"))],
        "batch_size": 144,
        "num_workers": 16,
        "pin_memory": True,
        "drop_last": True,
        "sample_segments": False,
        "sample_gene_masks": True,
    },
    "finetune": {
        "num_epochs": 20,
        "lr": 0.001,
        "start_lr": 1e-5,
        "final_lr": 1e-5,
        "warmup_epochs": 2,
        "weight_decay": 0.04,
        "final_weight_decay": 0.4,
        "ema_momentum": 0.9995,        # EMA for the target encoder
        "final_ema_momentum": 1.0,
        "loss_fn_type": "smooth_l1",
        "clip_grad": 2.0,
        "use_bfloat16": True,
        
        # LoRA (parameter-efficient fine-tuning)
        "use_peft": True,
        "peft_method": "lora",
        "peft_rank": 16,
        "peft_alpha": 256,
        "peft_dropout": 0.1,
        "peft_bias": "none",
        "peft_target_modules": ["qkv", "proj", "fc1", "fc2"],
        "save_every": 1,               # checkpoint every epoch
    },
}

**Important knobs**

- `pretrained_checkpoint_path`: the Xenium-pretrained TERRA model you are adapting.
- `finetune_dataset`: the list of tokenised sections from Step 2. They are concatenated and shuffled so batches mix sections.
- `use_peft=True` + `peft_*`: turn on LoRA and set its capacity. `peft_rank` (16) and `peft_alpha` (256) control adapter size/scaling; `peft_target_modules` chooses which layers get adapters (`qkv`, `proj`, `fc1`, `fc2` = attention + MLP).
- `ema_momentum`: how slowly the target encoder tracks the context encoder; higher = more stable targets.
- `num_epochs`, `lr`, `warmup_epochs`, `use_bfloat16`: standard training schedule; bf16 keeps memory down.

Now fine-tune. Passing `dataset=None` lets `finetune_self_supervised` load,
concatenate and shuffle the tokenised sections listed in the config for you:

In [ ]:
from datetime import datetime
from terra.training.finetune_self_supervised import finetune_self_supervised

# Only the LoRA adapters are trained (long GPU job). 
# dataset=None -> the tokenised sections in args["data"]["finetune_dataset"] are loaded,
# concatenated and shuffled automatically.
run_name = "finetune_" + datetime.now().strftime("%d%m%Y_%H%M%S")
run_dir = finetune_self_supervised(args=args, dataset=None, run_name=run_name)

:::{note}
Fine-tuning across many sections is a **multi-hour GPU job**.
:::

:::{warning}
Only the LoRA adapters are trainable, `finetune_self_supervised` asserts this
and will raise if the base weights were left unfrozen. When PEFT is on, the
target encoder's EMA update merges the adapters into the base weights on each
step, so the target encoder always holds fully-merged fine-tuned weights.
:::

## Step 4: Merge the LoRA adapters

The embedding API expects an ordinary model folder (`model_config.yaml`,
`token_dictionary.pkl`, `model_checkpoint.pt`), not a checkpoint with separate
LoRA adapters. `prepare_finetuned_model` folds the adapters into the base
weights (`W = W_base + B @ A`), copies the config and token dictionary from the
pretrained model, and writes a self-contained folder you can embed with.

In [ ]:
from terra.training.finetune_self_supervised import prepare_finetuned_model

# use_peft=True folds the LoRA adapters into the base weights (W = W_base + B @ A).
prepare_finetuned_model(
    finetuned_checkpoint_dir=run_dir,   # returned by finetune_self_supervised in Step 3
    pretrained_model_dir=PRETRAINED_MODEL_DIR,
    output_dir=str(FINETUNED_MODEL_DIR),
    use_peft=True,
)

# FINETUNED_MODEL_DIR now contains:
#   model_config.yaml, token_dictionary.pkl, model_checkpoint.pt, finetuning_metadata.yaml

:::{tip}
By default this merges the **last** epoch's checkpoint. Pass `checkpoint_epoch=N`
to prepare a specific epoch (e.g. the best-validation one).
:::

## Step 5: Embed sections with the adapted model

With the merged model in hand, embedding is the standard TERRA pipeline
(harmonize → tokenize → `embed_dataset`) but now pointed at
`FINETUNED_MODEL_DIR`. For each section we store the **`cell_emb`**
representation in `obsm`, the per-spot embedding. 

In [ ]:
def embed_section(h5ad_path, model_dir, out_dir, batch_size=128, num_workers=12):
    adata = sc.read_h5ad(h5ad_path)
    adata = harmonize_adata(adata)

    nan_mask = adata.obs[["x", "y"]].isna().any(axis=1)
    if nan_mask.any():
        adata = adata[~nan_mask].copy()

    dataset = tokenize_adata(
        adata,
        model_folder_path=model_dir,
        cache_directory_path=CACHE_DIR,
        nproc=4,
        processing_mode="parallel",
    )

    out = embed_dataset(
        dataset=dataset,
        model_folder_path=model_dir,
        batch_size=batch_size,
        num_workers=num_workers,
    )
    adata.obsm["cell_emb"] = out["cell_emb"]

    sample_name = Path(h5ad_path).stem.replace("_QC", "")
    out_path = Path(out_dir) / f"{sample_name}_adata_embeddings.h5ad"
    adata.write(out_path)
    print(f"  embedded -> {out_path}")
    return adata


for h5_file in sorted(ZS_QC_DIR.glob("*.h5ad")):
    print(f"Embedding {h5_file.stem}")
    embed_section(h5_file, str(FINETUNED_MODEL_DIR), ZS_EMBED_DIR)

## Step 6: Spatial mapping

Now we analyse the adapted-model embeddings across all sections together. We
merge the per-section embedding files, build a neighbor graph on the `cell_emb`
space, and run UMAP + Leiden to recover **cell types / states**, then map the
clusters back onto each tissue section.

### Merge the per-section embeddings

In [ ]:
adatas = []
for h5ad_file in sorted(ZS_EMBED_DIR.glob("*.h5ad")):
    if "ZS_merged" in h5ad_file.name:
        continue
    parts = h5ad_file.stem.split("_")
    sample_name = "_".join(p for p in parts
                           if p not in ["Visium", "Embryo", "adata", "embeddings"])
    a = sc.read_h5ad(h5ad_file)
    a.obs["sample"] = sample_name
    a.var_names_make_unique()
    adatas.append(a)

adata = ad.concat(adatas, join="inner", merge="unique")

# Keep ground truthspot annotations
adata = adata[~adata.obs["C_scANVI"].isna() & (adata.obs["C_scANVI"] != "NA")].copy()
adata.write_h5ad(ZS_EMBED_DIR / "ZS_merged_embeddings.h5ad")
print(adata)

### Cluster cell types on `cell_emb`

In [ ]:
sc.pp.neighbors(adata, use_rep="cell_emb", n_neighbors=15)
sc.tl.umap(adata, min_dist=0.5, spread=1.0)

sc.tl.leiden(adata, key_added="cell_leiden", resolution=0.15)

sc.pl.umap(adata, color=["sample", "C_scANVI", "cell_leiden"],
           size=2, frameon=False, wspace=0.4)

The `sample` panel is a **batch-integration** check: after adaptation, spots
from different sections should intermix rather than form section-specific
islands. The `C_scANVI` panel shows the known annotation, and `cell_leiden`
shows the unsupervised clusters recovered from the embedding alone.

### Map the clusters back onto the tissue

Plot the `cell_leiden` labels on each section's coordinates with `sc.pl.spatial` 

In [ ]:
for sample_name in adata.obs["sample"].unique():
    section = adata[adata.obs["sample"] == sample_name].copy()
    sc.pl.spatial(section, img_key=None, color="cell_leiden",
                  spot_size=300, frameon=False, title=str(sample_name))

---

## Questions / help

If you have any questions about this tutorial, please contact **Zichen He** — [zh4@sanger.ac.uk](mailto:zh4@sanger.ac.uk).